In [ ]:
!pip install pandas


[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


# PLN Projeto 1

In [ ]:
import pandas as pd
import re
import uuid

df_cases = pd.read_csv("../../data/external/cases.csv")
df_meta = pd.read_csv("../../data/external/metadata.csv")
# Cruzamento dos dados: Adiciona metadados aos casos
df_merged = df_cases.merge(df_meta, on='article_id', how='left')

In [ ]:
token_re = re.compile(r"""
    \d+(?:\.\d+)?(?:\s*x\s*10\d*)?
    |[a-zA-ZÀ-ÿ]+
    |[.,;:!?()%/]
""", re.VERBOSE)

def tokenize(sent):
    return [(m.group(), m.start(), m.end()) for m in token_re.finditer(sent)]

In [ ]:
def normalize(tok):
    return tok.lower().rstrip('sáéíóú')  # naive, mas cobre plural/acento simples

Lista da Wikipedia de sintomas médicos possíveis: https://pt.wikipedia.org/wiki/Lista_de_sintomas_médicos

In [ ]:
def build_gazetteers():
    return {
        'Diagnosis': ['diabetes mellitus', 'hypertension', 'acute pancreatitis', 'chronic pancreatitis', 'pancreatic pseudocyst', 'pseudocyst', 'solid mass', 'type 2 diabetes', 'tumor', 'cancer',
            'acute respiratory distress syndrome', 'sepsis', 'septic shock', 'pneumonia', 'pulmonary embolism', 'acute heart failure', 'congestive heart failure', 'myocardial infarction',
            'ischemic stroke', 'hemorrhagic stroke', 'acute kidney injury', 'chronic kidney disease', 'renal failure', 'acute liver failure', 'cirrhosis', 'hepatitis',
            'cholangitis', 'biliary obstruction', 'appendicitis', 'peritonitis', 'gastrointestinal bleeding', 'intestinal obstruction', 'peptic ulcer disease',
            'deep vein thrombosis', 'venous thromboembolism', 'anemia', 'thrombocytopenia', 'leukemia', 'lymphoma', 'hemolytic anemia', 'fatty liver disease',
            'metastatic disease', 'meningitis', 'encephalitis', 'pyelonephritis', 'urinary tract infection', 'bronchitis', 'asthma', 'pneumonitis', 'respiratory failure',
            'arrhythmia', 'atrial fibrillation', 'valvular disease', 'endocarditis', 'pericarditis', 'liver abscess', 'splenomegaly'
        ],
        'Symptom': ['epigastric pain', 'nausea', 'fever', 'epigastric tenderness', 'fat stranding', 'pain', 'headache', 'cough',
            'abdominal pain', 'vomiting', 'diarrhea', 'constipation', 'bloody stools', 'hematochezia', 'melena', 'shortness of breath', 'dyspnea', 'wheezing',
            'chest pain', 'palpitations', 'dizziness', 'syncope', 'fatigue', 'weakness', 'malaise', 'loss of appetite', 'anorexia', 'weight loss', 'weight gain',
            'swelling', 'edema', 'leg swelling', 'lower extremity edema', 'jaundice', 'dark urine', 'itching', 'pruritus', 'confusion', 'delirium', 'sleep disturbance',
            'insomnia', 'restlessness', 'anxiety', 'depression', 'tremor', 'muscle weakness', 'myalgia', 'arthralgia', 'back pain', 'joint pain', 'neck pain',
            'sore throat', 'hoarseness', 'hemoptysis', 'cyanosis', 'hematuria', 'dysuria', 'polyuria', 'oliguria', 'urinary retention', 'flank pain', 'cold intolerance',
            'heat intolerance', 'blurred vision', 'visual disturbance', 'vertigo', 'paresthesia', 'numbness', 'tingling', 'bruising', 'easy bruising', 'bleeding', 'nosebleed',
            'productive cough', 'nonproductive cough', 'dysphagia', 'odynophagia', 'hiccups', 'tachycardia', 'hypotension', 'hypertension symptoms', 'chills', 'rigors', 'sweating',
            'night sweats', 'burning sensation', 'pressure sensation', 'abdominal distension', 'bloating', 'gastroesophageal reflux', 'heartburn', 'dyspepsia', 'gastrointestinal pain', 'thirst',
            'polyuria', 'excessive thirst', 'hyperglycemia symptoms', 'hypoglycemia symptoms', 'fainting'
        ],
        'Exam': [
          'computed tomography', 'ct', 'thoracic ct', 'endoscopic ultrasound', 'eus', 'ultrasound', 'laboratory tests', 'mri', 'x-ray', 'echocardiogram', 'echocardiography', 'ecg', 'flow cytometry',
          'hemoglobin', 'hematocrit', 'white blood cell count', 'total leukocyte count', 'leucocytosis', 'lymphocyte count', 'platelet count', 'platelet counts', 'mean corpuscular volume', 
          'reticulocyte count', 'haptoglobin', 'adamts13 activity', 'c-reactive protein', 'crp', 'erythrocyte sedimentation rate', 'esr', 'creatinine', 'serum creatinine', 'albumin', 'total protein',
          'ferritin', 'lactate dehydrogenase', 'ldh', 'complement c3', 'complement c4', 'c3', 'c4', 'troponin', 'troponin i', 'ck-mb', 'creatine kinase', 'myoglobin', 'ejection fraction', 'lvef', 
          'mean gradient', 'nt-probnp', 'heart rate', 'blood pressure', 'mean arterial blood pressure', 'glycosylated hemoglobin', 'hemoglobin a1c', 'hba1c', 'blood sugar', 'immunoglobulin m', 
          'immunoglobulin g', 'igm', 'igg', 'igg levels', 'forced vital capacity', 'fvc', 'forced expiratory volume', 'fev1', 'diffusing capacity', 'dlco', 'oxygen saturation', 'respiratory rate', 
          'carcinoembryonic antigen', 'cea', 'pd-l1 tps score','hydroxychloroquine level', 'gtt',
        ],
        'Treatment': [
            'intravenous fluids', 'oral rehydration', 'analgesia', 'analgesics', 'pain control',
            'antibiotics', 'antiviral therapy', 'antifungal therapy', 'antiparasitic therapy',
            'anticoagulation', 'antiplatelet therapy', 'heparin', 'warfarin', 'enoxaparin',
            'insulin', 'metformin', 'glucose control', 'blood pressure control', 'antihypertensive therapy',
            'fluid resuscitation', 'electrolyte replacement', 'oxygen therapy', 'ventilation support',
            'corticosteroids', 'prednisone', 'hydrocortisone', 'dexamethasone', 'steroid therapy',
            'immunosuppressive therapy', 'plasma exchange', 'plasmapheresis', 'intravenous immunoglobulin', 'ivig',
            'chemotherapy', 'radiotherapy', 'targeted therapy', 'immunotherapy', 'hormonal therapy',
            'surgery', 'laparotomy', 'laparoscopy', 'resection', 'drainage', 'debridement',
            'endoscopic drainage', 'cystgastrostomy', 'gastrojejunostomy', 'stent placement', 'biliary drainage',
            'dialysis', 'renal replacement therapy', 'liver transplant', 'kidney transplant', 'transplantation',
            'blood transfusion', 'platelet transfusion', 'erythrocyte transfusion',
            'physical therapy', 'rehabilitation', 'respiratory physiotherapy', 'occupational therapy',
            'bronchodilator therapy', 'inhaled beta agonist', 'nebulized therapy', 'smoking cessation',
            'prophylactic antibiotics', 'empiric therapy', 'supportive care', 'palliative care',
            'nutritional support', 'enteral nutrition', 'parenteral nutrition', 'oral nutrition',
            'thrombolysis', 'vasopressor support', 'inotropic support', 'antiemetics', 'anti-inflammatory therapy',
            'antihistamines', 'acid suppression', 'proton pump inhibitor', 'ppi', 'antacids',
            'beta-blocker therapy', 'statin therapy', 'aspirin', 'ibuprofen', 'acetaminophen', 'paracetamol',
            'antidepressant therapy', 'antiepileptic therapy', 'antiarrhythmic therapy', 'diuretic therapy',
            'vaccination', 'immunization', 'monitoring', 'observation', 'watchful waiting', 'conservative management'
        ]
    }

In [ ]:
def build_lookup(gazetteer):
    lookup = {}
    for ent_type, keywords in gazetteer.items():
        for kw in keywords:
            key = tuple(normalize(t[0]) for t in tokenize(kw))
            lookup[key] = (ent_type, kw)
    print(lookup)
    return lookup

In [ ]:
gazetteer = build_gazetteers()
lookup = build_lookup(gazetteer)

{'diabete': ('Diagnosis', 'type 2 diabetes'), 'mellitu': ('Diagnosis', 'diabetes mellitus'), 'hypertension': ('Symptom', 'hypertension symptoms'), 'acute': ('Diagnosis', 'acute liver failure'), 'pancreatiti': ('Diagnosis', 'chronic pancreatitis'), 'chronic': ('Diagnosis', 'chronic kidney disease'), 'pancreatic': ('Diagnosis', 'pancreatic pseudocyst'), 'pseudocyst': ('Diagnosis', 'pseudocyst'), 'solid': ('Diagnosis', 'solid mass'), 'ma': ('Diagnosis', 'solid mass'), 'type': ('Diagnosis', 'type 2 diabetes'), '2': ('Diagnosis', 'type 2 diabetes'), 'tumor': ('Diagnosis', 'tumor'), 'cancer': ('Diagnosis', 'cancer'), 'respiratory': ('Treatment', 'respiratory physiotherapy'), 'distre': ('Diagnosis', 'acute respiratory distress syndrome'), 'syndrome': ('Diagnosis', 'acute respiratory distress syndrome'), 'sepsi': ('Diagnosis', 'sepsis'), 'septic': ('Diagnosis', 'septic shock'), 'shock': ('Diagnosis', 'septic shock'), 'pneumonia': ('Diagnosis', 'pneumonia'), 'pulmonary': ('Diagnosis', 'pulmonar

In [ ]:
def create_node(all_nodes, node_id, node_type, node_label, **attributes):
    node = {
        "node_id": node_id,
        "type": node_type,
        "label": node_label,
        "attributes": attributes,
    }
    all_nodes.append(node)
    return node

In [ ]:
def create_edge(all_edges, edge_id, source_id, target_id, edge_type, relation, **attributes):
    edge = {
        "edge_id": edge_id,
        "source": source_id,
        "target": target_id,
        "type": edge_type,
        "relation": relation,
        "attributes": attributes,
    }
    all_edges.append(edge)
    return edge

In [ ]:
def match_entities(tokens, lookup):
    n = len(tokens)
    consumed = [False] * n
    matches = []
    max_n = max(len(k) for k in lookup)

    for size in range(max_n, 0, -1):
        i = 0
        while i + size <= n:
            if any(consumed[i:i+size]):
                i += 1
                continue
            key = tuple(normalize(tokens[k][0]) for k in range(i, i + size))
            if key in lookup:
                ent_type, keyword = lookup[key]
                matches.append({'start': i, 'end': i + size, 'ent_type': ent_type, 'keyword': keyword})
                for k in range(i, i + size):
                    consumed[k] = True
                i += size
            else:
                i += 1
    matches.sort(key=lambda m: m['start'])
    return matches

In [ ]:
def process_multicare_dataset(row, lookup):
    all_nodes = []
    all_edges = []
    seen_edges = set()

    case_id = row["case_id"]
    text = row["case_text"]

    # Creates patient node
    patient_id = f"P_{case_id}"
    age = row.get('age', 'Unknown')
    gender = row.get('gender', 'Unknown')
    create_node(all_nodes, patient_id, 'Patient', f'Case {case_id}', age=age, gender=gender)

    entity_map = {}
    valid_units = {'u/l', 'mg/l', 'ng/ml', 'mmol/l', 'g/dl', '%', 'au/ml', 'mmhg', 'l'}
    # Regex for numbers and scientific notation
    num_re = re.compile(r'\d+(?:\.\d+)?(?:\s*x\s*10\d*)?')

    sentences = re.split(r'(?<=[.!?])\s+', text)

    for sent in sentences:
        tokens = tokenize(sent)
        last_node_id = patient_id
        last_type = None

        for m in match_entities(tokens, lookup):
            ent_type, keyword = m['ent_type'], m['keyword']

            if keyword not in entity_map:
                node_id = f"{ent_type[:3].upper()}_{uuid.uuid4().hex[:6]}"
                create_node(all_nodes, node_id, ent_type, keyword.capitalize())
                entity_map[keyword] = node_id

            node_id = entity_map[keyword]

            relation = 'ASSOCIATED_WITH'
            if ent_type == 'Diagnosis': relation = 'DIAGNOSED_WITH'
            elif ent_type == 'Symptom': relation = 'HAS_SYMPTOM'
            elif ent_type == 'Exam': relation = 'UNDERWENT_EXAM'
            elif ent_type == 'Treatment': relation = 'TREATED_BY'

            edge_key = (patient_id, node_id, relation)
            if edge_key not in seen_edges:
                seen_edges.add(edge_key)
                create_edge(all_edges, f"e_{uuid.uuid4().hex[:6]}",
                            patient_id, node_id, ent_type, relation, source='regex_dict')

            if ent_type == 'Exam':
                last_node_id, last_type = node_id, 'Exam'

                for j in range(i + 1, len(tokens)):
                    next_tok = tokens[j][0]
                    if normalize(next_tok) in lookup:
                        break
                    if num_re.fullmatch(next_tok) and j + 1 < len(tokens):
                        unit_tok = tokens[j + 1][0].lower().lstrip('/')
                        if unit_tok in valid_units:
                            res_id = f"VAL_{uuid.uuid4().hex[:6]}"
                            create_node(all_nodes, res_id, 'ExamResult',
                                        f'{next_tok} {unit_tok}', value=next_tok, unit=unit_tok)

                            exam_edge_key = (last_node_id, res_id, 'HAS_RESULT')
                            if exam_edge_key not in seen_edges:
                                seen_edges.add(exam_edge_key)
                                create_edge(all_edges, f"e_{uuid.uuid4().hex[:6]}",
                                            last_node_id, res_id, 'ExamResult',
                                            'HAS_RESULT', source='regex_extraction')
                            break

    return pd.DataFrame(all_nodes), pd.DataFrame(all_edges)

In [ ]:
df_nodes, df_edges = process_multicare_dataset(df_merged.iloc[6], lookup)  # Processa apenas o caso de índice 10 como exemplo)
df_nodes.to_csv("../../data/processed/nodes.csv", index=False)
df_edges.to_csv("../../data/processed/edges.csv", index=False)
print(f"Extração em lote concluída! {len(df_nodes)} nós e {len(df_edges)} arestas criadas.")

Extração em lote concluída! 58 nós e 57 arestas criadas.


In [ ]:
def to_mermaid(nodes, edges) -> str:
    lines = ["flowchart LR"]
    for n in nodes.itertuples():
        label = str(n.label).replace('"', "'")
        lines.append(f'  {n.node_id}["{n.type}<br/>{label}"]')
    for e in edges.itertuples():
        lines.append(f'  {e.source} -->|{e.relation}| {e.target}')
    return "\n".join(lines)

In [ ]:
from IPython.display import display, Markdown

mermaid_md = to_mermaid(df_nodes, df_edges)
display(Markdown(f"```mermaid\n{mermaid_md}\n```"))

```mermaid
flowchart LR
  P_PMC4835621_01["Patient<br/>Case PMC4835621_01"]
  SYM_2c67a9["Symptom<br/>Lower extremity edema"]
  SYM_6b7b4e["Symptom<br/>Headache"]
  SYM_33ea13["Symptom<br/>Fatigue"]
  SYM_d65392["Symptom<br/>Malaise"]
  SYM_47528b["Symptom<br/>Fever"]
  SYM_70f3ac["Symptom<br/>Weight loss"]
  SYM_2319d6["Symptom<br/>Weight gain"]
  SYM_743d50["Symptom<br/>Night sweats"]
  SYM_9ee996["Symptom<br/>Hematuria"]
  SYM_babcd9["Symptom<br/>Diarrhea"]
  SYM_ce043c["Symptom<br/>Nausea"]
  SYM_ac3f15["Symptom<br/>Vomiting"]
  SYM_39278c["Symptom<br/>Bloody stools"]
  SYM_9580b7["Symptom<br/>Hypertension symptoms"]
  TRE_d90def["Treatment<br/>Beta-blocker therapy"]
  TRE_106a12["Treatment<br/>Physical therapy"]
  EXA_55f55b["Exam<br/>Laboratory tests"]
  EXA_0a3927["Exam<br/>Forced vital capacity"]
  DIA_1e2f0a["Diagnosis<br/>Acute respiratory distress syndrome"]
  SYM_4d655a["Symptom<br/>Jaundice"]
  EXA_8bacd5["Exam<br/>Heart rate"]
  DIA_1ae6e8["Diagnosis<br/>Splenomegaly"]
  SYM_5e7632["Symptom<br/>Melena"]
  SYM_254d07["Symptom<br/>Hematochezia"]
  EXA_7bacdc["Exam<br/>Thoracic ct"]
  SYM_67f267["Symptom<br/>Bleeding"]
  EXA_21eed1["Exam<br/>White blood cell count"]
  TRE_0a7ab9["Treatment<br/>Blood transfusion"]
  EXA_5d2841["Exam<br/>Reticulocyte count"]
  VAL_70a596["ExamResult<br/>73 %"]
  EXA_6d9e30["Exam<br/>Hemoglobin a1c"]
  EXA_fb7b0a["Exam<br/>Immunoglobulin g"]
  EXA_9d87ba["Exam<br/>Mean arterial blood pressure"]
  EXA_162d9d["Exam<br/>Mean corpuscular volume"]
  EXA_48989a["Exam<br/>Forced expiratory volume"]
  VAL_c41449["ExamResult<br/>2.4 %"]
  EXA_fd6df4["Exam<br/>Hydroxychloroquine level"]
  EXA_1c466c["Exam<br/>Serum creatinine"]
  TRE_0e388c["Treatment<br/>Liver transplant"]
  EXA_ed657d["Exam<br/>Lactate dehydrogenase"]
  EXA_3f9415["Exam<br/>Ldh"]
  EXA_ba0b79["Exam<br/>Pd-l1 tps score"]
  EXA_bf8abc["Exam<br/>Haptoglobin"]
  EXA_c54b45["Exam<br/>Adamts13 activity"]
  VAL_886b7b["ExamResult<br/>10 %"]
  TRE_ef1ccc["Treatment<br/>Plasmapheresis"]
  TRE_7cadfa["Treatment<br/>Steroid therapy"]
  EXA_075c96["Exam<br/>Computed tomography"]
  SYM_790d8b["Symptom<br/>Chest pain"]
  TRE_609702["Treatment<br/>Platelet transfusion"]
  DIA_dce21c["Diagnosis<br/>Deep vein thrombosis"]
  TRE_544230["Treatment<br/>Heparin"]
  DIA_782141["Diagnosis<br/>Chronic kidney disease"]
  TRE_10ad1a["Treatment<br/>Diuretic therapy"]
  TRE_7db0fd["Treatment<br/>Prophylactic antibiotics"]
  DIA_a1bba3["Diagnosis<br/>Type 2 diabetes"]
  SYM_ba07d6["Symptom<br/>Hypoglycemia symptoms"]
  P_PMC4835621_01 -->|HAS_SYMPTOM| SYM_2c67a9
  P_PMC4835621_01 -->|HAS_SYMPTOM| SYM_6b7b4e
  P_PMC4835621_01 -->|HAS_SYMPTOM| SYM_33ea13
  P_PMC4835621_01 -->|HAS_SYMPTOM| SYM_d65392
  P_PMC4835621_01 -->|HAS_SYMPTOM| SYM_47528b
  P_PMC4835621_01 -->|HAS_SYMPTOM| SYM_70f3ac
  P_PMC4835621_01 -->|HAS_SYMPTOM| SYM_2319d6
  P_PMC4835621_01 -->|HAS_SYMPTOM| SYM_743d50
  P_PMC4835621_01 -->|HAS_SYMPTOM| SYM_9ee996
  P_PMC4835621_01 -->|HAS_SYMPTOM| SYM_babcd9
  P_PMC4835621_01 -->|HAS_SYMPTOM| SYM_ce043c
  P_PMC4835621_01 -->|HAS_SYMPTOM| SYM_ac3f15
  P_PMC4835621_01 -->|HAS_SYMPTOM| SYM_39278c
  P_PMC4835621_01 -->|HAS_SYMPTOM| SYM_9580b7
  P_PMC4835621_01 -->|TREATED_BY| TRE_d90def
  P_PMC4835621_01 -->|TREATED_BY| TRE_106a12
  P_PMC4835621_01 -->|UNDERWENT_EXAM| EXA_55f55b
  P_PMC4835621_01 -->|UNDERWENT_EXAM| EXA_0a3927
  P_PMC4835621_01 -->|DIAGNOSED_WITH| DIA_1e2f0a
  P_PMC4835621_01 -->|HAS_SYMPTOM| SYM_4d655a
  P_PMC4835621_01 -->|UNDERWENT_EXAM| EXA_8bacd5
  P_PMC4835621_01 -->|DIAGNOSED_WITH| DIA_1ae6e8
  P_PMC4835621_01 -->|HAS_SYMPTOM| SYM_5e7632
  P_PMC4835621_01 -->|HAS_SYMPTOM| SYM_254d07
  P_PMC4835621_01 -->|UNDERWENT_EXAM| EXA_7bacdc
  P_PMC4835621_01 -->|HAS_SYMPTOM| SYM_67f267
  P_PMC4835621_01 -->|UNDERWENT_EXAM| EXA_21eed1
  P_PMC4835621_01 -->|TREATED_BY| TRE_0a7ab9
  P_PMC4835621_01 -->|UNDERWENT_EXAM| EXA_5d2841
  EXA_5d2841 -->|HAS_RESULT| VAL_70a596
  P_PMC4835621_01 -->|UNDERWENT_EXAM| EXA_6d9e30
  P_PMC4835621_01 -->|UNDERWENT_EXAM| EXA_fb7b0a
  P_PMC4835621_01 -->|UNDERWENT_EXAM| EXA_9d87ba
  P_PMC4835621_01 -->|UNDERWENT_EXAM| EXA_162d9d
  P_PMC4835621_01 -->|UNDERWENT_EXAM| EXA_48989a
  EXA_5d2841 -->|HAS_RESULT| VAL_c41449
  P_PMC4835621_01 -->|UNDERWENT_EXAM| EXA_fd6df4
  P_PMC4835621_01 -->|UNDERWENT_EXAM| EXA_1c466c
  P_PMC4835621_01 -->|TREATED_BY| TRE_0e388c
  P_PMC4835621_01 -->|UNDERWENT_EXAM| EXA_ed657d
  P_PMC4835621_01 -->|UNDERWENT_EXAM| EXA_3f9415
  P_PMC4835621_01 -->|UNDERWENT_EXAM| EXA_ba0b79
  P_PMC4835621_01 -->|UNDERWENT_EXAM| EXA_bf8abc
  P_PMC4835621_01 -->|UNDERWENT_EXAM| EXA_c54b45
  EXA_55f55b -->|HAS_RESULT| VAL_886b7b
  P_PMC4835621_01 -->|TREATED_BY| TRE_ef1ccc
  P_PMC4835621_01 -->|TREATED_BY| TRE_7cadfa
  P_PMC4835621_01 -->|UNDERWENT_EXAM| EXA_075c96
  P_PMC4835621_01 -->|HAS_SYMPTOM| SYM_790d8b
  P_PMC4835621_01 -->|TREATED_BY| TRE_609702
  P_PMC4835621_01 -->|DIAGNOSED_WITH| DIA_dce21c
  P_PMC4835621_01 -->|TREATED_BY| TRE_544230
  P_PMC4835621_01 -->|DIAGNOSED_WITH| DIA_782141
  P_PMC4835621_01 -->|TREATED_BY| TRE_10ad1a
  P_PMC4835621_01 -->|TREATED_BY| TRE_7db0fd
  P_PMC4835621_01 -->|DIAGNOSED_WITH| DIA_a1bba3
  P_PMC4835621_01 -->|HAS_SYMPTOM| SYM_ba07d6
```